1. Task Overview and Learning Objectives

1. Task Overview and Learning Objectives
This assignment represents a critical transition in the modern data engineering and machine
learning lifecycle. You will migrate from classical machine learning algorithms to deep learning
architectures utilizing Feedforward Neural Networks (Multilayer Perceptrons). Furthermore, you
will abandon manual file uploads in favor of a fully programmatic Extract, Transform, Load, and
Serve (ETLS) deployment pipeline.
By the conclusion of this task, you will be required to:
1. Programmatically synchronize your local computational environment with a remote
GitHub repository.
2. Engineer, compile, and train two distinct Neural Network topologies for binary
classification on the Titanic dataset.
3. Automatically generate markdown documentation (README.md) utilizing Python File I/O.
4. Deploy the generated model artifacts (.h5), codebase, and documentation simultaneously
to a data lake (AWS S3) and a version control serving layer (GitHub).
5. Append execution logs to an ongoing continuous integration ledger (AUDIT_TRAIL.md)

In [3]:
!pip install GitPython PyGithub boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.9 MB/s eta 0:00:00


In [4]:
from github import Github, GithubException, Auth
from datetime import datetime
from google.colab import userdata

# GitHub Credentials and Targeting
STUDENT_TOKEN = userdata.get('STUDENT_TOKEN')
REPO_NAME = "nikk909/mds7-YueMa"
TARGET_FOLDER = "week-05-06-bigquery/deeplearning"
clean_filename = "titanic_clean.csv"

In [5]:
# Authenticate with GitHub (Updated Auth Method)
auth = Auth.Token(STUDENT_TOKEN)
g = Github(auth=auth)
repo = g.get_repo(REPO_NAME)
print(f"Authenticated to repository: {repo.full_name}")

Authenticated to repository: nikk909/mds7-YueMa


Phase 2: Data Extraction & Preprocessing


● Locate the previously engineered titanic_clean.csv (either from the cloned repository or via a direct boto3 GET request to your AWS S3 bucket).
● Isolate your feature matrix (X) and target vector (y - Survived).
● Critical Requirement: Neural networks are highly sensitive to unscaled input data. You must apply a StandardScaler or MinMaxScaler to your feature matrix prior to training to ensure gradient convergence.

In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Data Extraction (Phase 2 - Step 1)
DATA_URL = "https://raw.githubusercontent.com/nikk909/mds7-YueMa/refs/heads/main/week-03-04-powerbi/titanic_clean.csv"
df = pd.read_csv(DATA_URL)

# 2. Isolate Target Vector (y)
y = df['Survived']

# 3. Isolate Feature Matrix (X) -
# We select only the columns that represent the features of a passenger
# excluding metadata (Names/IDs) and the target itself.
feature_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
X = df[feature_cols]

# 4. Apply Scaling (Critical Requirement) -
# Now X is a pure numeric matrix, we can strictly follow the scaling instruction.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Dataset Splitting
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Feature Matrix X isolated with columns: {feature_cols}")

Feature Matrix X isolated with columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']


Phase 3: Neural Network Engineering & Training

You are required to construct two distinct neural network models using the Keras Sequential
API. Both models must utilize the adam optimizer and the binary_crossentropy loss function
suitable for binary classification.
Model A: Shallow Network (3 Layers)
● Architecture: 1 Input/Hidden Layer, 1 Output Layer.
● Activation: Use ReLU for the hidden layer and Sigmoid for the output layer.
Model B: Deep Network (5 Layers)
● Architecture: 1 Input Layer, 3 distinct Hidden Layers, 1 Output Layer.
● Activation: Use ReLU for all hidden layers to mitigate the vanishing gradient problem,
and Sigmoid for the output layer.
Artifact Generation:
● Train both models for an identical number of epochs (e.g., 50 or 100).
● Evaluate both models against your test split and record the accuracy metrics.
● Save both computational graphs and their respective weights to the local Colab disk
strictly utilizing the Keras Hierarchical Data Format. Name them model_3_layers.h5 and
model_5_layers.h5.

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Get the number of input features
input_dim = X_train.shape[1]

# --- 1. Model A: Shallow Network (3 Layers) ---
# Architecture: 1 Input/Hidden Layer, 1 Output Layer
model_a = Sequential([
    # Input/Hidden Layer: Use ReLU
    Dense(16, activation='relu', input_shape=(input_dim,)),
    # Output Layer: Use Sigmoid for binary classification
    Dense(1, activation='sigmoid')
])

model_a.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 2. Model B: Deep Network (5 Layers) ---
# Architecture: 1 Input Layer, 3 distinct Hidden Layers, 1 Output Layer
model_b = Sequential([
    # Input Layer
    Dense(32, activation='relu', input_shape=(input_dim,)),
    # 3 Hidden Layers: Use ReLU to mitigate vanishing gradient
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    # Output Layer: Use Sigmoid
    Dense(1, activation='sigmoid')
])

model_b.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 3. Training & Artifact Generation ---
# Train both for an identical number of epochs (e.g., 50)
print("Training Model A...")
history_a = model_a.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
loss_a, acc_a = model_a.evaluate(X_test, y_test, verbose=0)

print("Training Model B...")
history_b = model_b.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
loss_b, acc_b = model_b.evaluate(X_test, y_test, verbose=0)

# --- 4. Save Models ---
# Strictly utilize Keras HDF5 format (.h5)
model_a.save("model_3_layers.h5")
model_b.save("model_5_layers.h5")

print(f"\nModel A Test Accuracy: {acc_a:.4f}")
print(f"Model B Test Accuracy: {acc_b:.4f}")
print("Models saved as .h5 files.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Model A...
Training Model B...



Model A Test Accuracy: 0.7933
Model B Test Accuracy: 0.7989
Models saved as .h5 files.


Phase 4: Automated Documentation Generation

Data pipelines must be self-documenting. You must utilize Python's native file handling
capabilities to write a README.md file dynamically.
● The script must generate a string containing markdown syntax.
● The documentation must explicitly detail the topology of Model A and Model B (layer
counts, neuron densities, activation functions).
● The document must include the final test accuracy achieved by both models.
● Save this file locally in the Colab runtime as README.md.

In [8]:
# --- Phase 4: Automated Documentation Generation ---

# 1. 定义 Markdown 内容字符串
# 必须包含模型拓扑结构（层数、神经元密度、激活函数）和测试准确率
readme_content = f"""# Titanic Deep Learning Pipeline Report

## 1. Model Architectures
This pipeline evaluates two distinct Neural Network topologies for binary classification.

### Model A: Shallow Network (3 Layers)
- **Structure**: 1 Input/Hidden Layer, 1 Output Layer.
- **Neuron Density**: 16 neurons in the hidden layer.
- **Activation Functions**: ReLU for hidden, Sigmoid for output.
- **Final Test Accuracy**: {acc_a:.4f}

### Model B: Deep Network (5 Layers)
- **Structure**: 1 Input Layer, 3 Hidden Layers, 1 Output Layer.
- **Neuron Density**: 32, 16, 8 neurons respectively in hidden layers.
- **Activation Functions**: ReLU for all hidden layers, Sigmoid for output.
- **Final Test Accuracy**: {acc_b:.4f}

## 2. Pipeline Execution Details
- **Optimizer**: Adam
- **Loss Function**: Binary Crossentropy
- **Framework**: Keras Sequential API
- **Deployment**: Automated dual-cloud upload to AWS S3 and GitHub.
"""

# 2. 使用 Python 原生文件处理写入本地磁盘
# 文件名必须严格为 README.md
with open("README.md", "w") as f:
    f.write(readme_content)

print("Successfully generated README.md locally.")

Successfully generated README.md locally.


Phase 5: Dual-Cloud Artifact Deployment

Your generated artifacts (the two .h5 models, the executed .ipynb notebook, and the generated README.md) must now be deployed to your cloud infrastructure.
Part A: AWS S3 Data Lake (The Load Layer)
● Initialize the boto3 client targeting the eu-north-1 region.
● Upload all four artifacts to your awsstoragecloud bucket.
● You must store these within a logical folder structure: deeplearning/.
Part B: GitHub Serving Layer (The Serve Layer)
● Initialize the PyGithub client using your Personal Access Token.
● Target the specific directory: week-05-06-bigquery/deeplearning/.
● Programmatically execute repo.create_file() (or update_file() if overwriting) to commit the
models, notebook, and README to the repository.
● Note: GitHub has strict file size limits; ensure your .h5 files do not exceed 50MB, which
should not be an issue for the Titanic dataset.

In [ ]:
# --- Pre-deployment: Sync current notebook to local disk ---
from google.colab import drive
import shutil

# 1. Mount Google Drive (if not already mounted)
drive.mount('/content/drive')

# 2. Define the source path in Drive and the target path in local runtime
# Note: Ensure the path matches where your file is stored in Google Drive
drive_path = '/content/drive/MyDrive/Colab Notebooks/data science innovations week5-6.ipynb'
local_path = 'data science innovations week5-6.ipynb'

# 3. Copy the file from Drive to the current local environment
try:
    shutil.copy(drive_path, local_path)
    print(f"Successfully synced '{local_path}' from Drive to local runtime.")
except FileNotFoundError:
    print("Error: Could not find the file in Drive. Please check the 'drive_path' variable.")

In [9]:
import boto3
import os
import glob
from google.colab import userdata

# --- Part A: AWS S3 Configuration ---
s3_client = boto3.client(
    's3',
    aws_access_key_id=userdata.get('AWS_ACCESS_KEY'),
    aws_secret_access_key=userdata.get('AWS_SECRET_KEY'),
    region_name='eu-north-1'
)

BUCKET_NAME = 'yuemadatascienceannovationsproject1'

In [10]:
import os
# --- Part B: Artifact Discovery & Unified Deployment ---

current_notebook = 'data science innovations week5-6.ipynb'
FILES_TO_DEPLOY = ['model_3_layers.h5', 'model_5_layers.h5', 'README.md', current_notebook]

# --- Step 1: AWS S3 Deployment (Data Lake) ---
print(f"--- Starting Step 1: AWS S3 Deployment ---")
for file_name in FILES_TO_DEPLOY:
    if os.path.exists(file_name):
        s3_path = f"deeplearning/{file_name}"
        s3_client.upload_file(file_name, BUCKET_NAME, s3_path)
        print(f"S3: Successfully uploaded {file_name}")
    else:
        print(f"S3 Warning: {file_name} not found locally.")

# --- Step 2: GitHub Deployment (Serving Layer) ---
print(f"\n--- Starting Step 2: GitHub Deployment ---")
for file_name in FILES_TO_DEPLOY:
    if os.path.exists(file_name):
        with open(file_name, 'rb') as f:
            file_content = f.read()

        git_path = f"{TARGET_FOLDER}/{file_name}"
        try:
            # Reusing 'repo' and 'TARGET_FOLDER' from your existing session
            git_obj = repo.get_contents(git_path)
            repo.update_file(git_path, f"Update {file_name}", file_content, git_obj.sha)
            print(f"GitHub: Successfully updated {file_name}")
        except Exception:
            repo.create_file(git_path, f"Initial deploy of {file_name}", file_content)
            print(f"GitHub: Successfully created {file_name}")

--- Starting Step 1: AWS S3 Deployment ---
S3: Successfully uploaded model_3_layers.h5
S3: Successfully uploaded model_5_layers.h5
S3: Successfully uploaded README.md
S3 Warning: data science innovations week5-6.ipynb not found locally.

--- Starting Step 2: GitHub Deployment ---
GitHub: Successfully updated model_3_layers.h5
GitHub: Successfully updated model_5_layers.h5
GitHub: Successfully updated README.md
